# EDA - Squad 08

### Leitura das bases

In [ ]:
# Criar dicionário para armazenar DataFrames
dataframes = {}
caminho_root = '/Volumes/hackathon_2025/default/source/'

# base_dados_cadastrais
try:
    df = spark.read.parquet(f'{caminho_root}base_dados_cadastrais/')
    dataframes['base_dados_cadastrais'] = df
    print(f'base_dados_cadastrais:{df.count():,} linhas, {len(df.columns)} colunas')
except Exception as e:
    print(f'Erro: {str(e)[:100]}')

# base_score_bureau_movel
try:
    df = spark.read.parquet(f'{caminho_root}base_score_bureau_movel/')
    dataframes['base_score_bureau_movel'] = df
    print(f'base_score_bureau_movel:{df.count():,} linhas, {len(df.columns)} colunas')
except Exception as e:
    print(f'Erro: {str(e)[:100]}')

# base_score_bureau_movel_full
try:
    df = spark.read.parquet(f'{caminho_root}base_score_bureau_movel_full/')
    dataframes['base_score_bureau_movel_full'] = df
    print(f'base_score_bureau_movel_full:{df.count():,} linhas, {len(df.columns)} colunas')
except Exception as e:
    print(f'Erro: {str(e)[:100]}')

# base_telco
try:
    df = spark.read.parquet(f'{caminho_root}base_telco/')
    dataframes['base_telco'] = df
    print(f'base_telco:{df.count():,} linhas, {len(df.columns)} colunas')
except Exception as e:
    print(f'Erro: {str(e)[:100]}')

# csv de bases_recarga
csv_files = [
    'BI_DIM_CANAL_AQUISICAO_CREDITO',
    'BI_DIM_FORMA_PAGAMENTO',
    'BI_DIM_INSTITUICAO',
    'BI_DIM_PLANO_PRECO',
    'BI_DIM_PLATAFORMA',
    'BI_DIM_PROMOCAO_CREDITO',
    'BI_DIM_STATUS_PLATAFORMA',
    'BI_DIM_TECNOLOGIA',
    'BI_DIM_TIPO_CREDITO',
    'BI_DIM_TIPO_INSERCAO',
    'BI_DIM_TIPO_RECARGA'
]

for csv_file in csv_files:
    try:
        df = spark.read.csv(f'{caminho_root}bases_recarga/{csv_file}.csv', header=True, inferSchema=True)
        dataframes[csv_file] = df
        print(f'{csv_file}: {df.count():,} linhas')
    except Exception as e:
        print(f'{csv_file}: {str(e)[:50]}')

# BI_DIM_TIPO_FATURAMENTO
try:
    df = spark.read.csv(f'{caminho_root}book_atraso/BI_DIM_TIPO_FATURAMENTO.csv', header=True, inferSchema=True)
    dataframes['BI_DIM_TIPO_FATURAMENTO'] = df
    print(f'BI_DIM_TIPO_FATURAMENTO: {df.count():,} linhas')
except Exception as e:
    print(f'Erro: {str(e)[:100]}')

# parquet - book_atraso/dados_faturamento/
try:
    df = spark.read.parquet(f'{caminho_root}book_atraso/dados_faturamento/')
    dataframes['dados_faturamento'] = df
    print(f'dados_faturamento: {df.count():,} linhas, {len(df.columns)} colunas')
except Exception as e:
    print(f'Erro: {str(e)[:100]}')

# parquet - bases_recarga/BI_FP_ASS_RECARGA_CMV_NOVA/
try:
    df = spark.read.parquet(f'{caminho_root}bases_recarga/BI_FP_ASS_RECARGA_CMV_NOVA/')
    dataframes['BI_FP_ASS_RECARGA_CMV_NOVA'] = df
    print(f'BI_FP_ASS_RECARGA_CMV_NOVA: {df.count():,} linhas, {len(df.columns)} colunas')
except Exception as e:
    print(f'Erro: {str(e)[:100]}')

# parquet - book_pagamento/dados_pagamento/
import os
caminho_pagamento = f'{caminho_root}book_pagamento/dados_pagamento/'

try:
    # Tentar carregar com wildcard
    df = spark.read.parquet(f'{caminho_pagamento}*.parquet')
    dataframes['dados_pagamento'] = df
    print(f'dados_pagamento (wildcard): {df.count():,} linhas, {len(df.columns)} colunas')
except Exception as e:
    print(f'Wildcard falhou, tentando listar arquivos...')

    # Se wildcard falhar, tentar carregar cada arquivo
    try:
        arquivos = os.listdir(caminho_pagamento)
        print(f'Encontrados {len(arquivos)} arquivos:')

        for arquivo in arquivos:
            if arquivo.endswith('.parquet'):
                print(f"     → {arquivo}")
                try:
                    caminho_completo = os.path.join(caminho_pagamento, arquivo)
                    df = spark.read.parquet(caminho_completo)

                    # Usar nome do arquivo como chave
                    nome_tabela = arquivo.replace('.parquet', '').replace('-c000', '')
                    if nome_tabela not in dataframes:  # Evitar duplicatas
                        dataframes[nome_tabela] = df

                    print(f'{nome_tabela}: {df.count():,} linhas')
                except Exception as e2:
                    print(f'Erro: {str(e2)[:50]}')
    except Exception as e:
        print(f'Erro ao listar diretório: {str(e)[:100]}')

# Converter todos os nomes dos dataframes para minúsculas
dataframes = {chave.lower(): df for chave, df in dataframes.items()}

### Metadados

In [ ]:
# importar biblioteca
from pyspark.sql.functions import col, count, when, lit


# Lista para armazenar metadados
metadata = []

# Iterar sobre cada dataframe
for nome_df, df in dataframes.items():
    total_linhas = df.count()

    # Analisar cada coluna
    for coluna in df.columns:
        tipo = df.schema[coluna].dataType

        # Contar nulos
        qt_nulos = df.filter(col(coluna).isNull()).count()
        percent_nulos = (qt_nulos / total_linhas) * 100 if total_linhas > 0 else 0

        # Cardinalidade (valores únicos)
        cardinalidade = df.select(coluna).distinct().count()

        # Adicionar à lista
        metadata.append({
            'nome_dataframe': nome_df,
            'nome_variavel': coluna,
            'tipo': str(tipo),
            'qt_nulos': int(qt_nulos),
            'percent_nulos': round(percent_nulos, 2),
            'cardinalidade': int(cardinalidade)
        })

# Converter para Spark DataFrame
df_resultado = spark.createDataFrame(metadata)

# Exibir
display(df_resultado)

### Tratamento dos dados

Converter os tipo de variáveis
- Datas: DATADENASCIMENTO (base_dados_cadastrais) para date e 59 colunas para datetime
- Valores: 27 colunas para float
- Flags e score: 13 colunas para integer
- safra: 4 colunas YYYYMM para YYYY/MM

In [ ]:
from pyspark.sql.functions import to_date, col, substring, concat, lit

total_dat = 0
total_datadenascimento = 0
total_val = 0
total_safra = 0
total_int = 0

# Lista de colunas para converter em integer
colunas_integer = [
    'FLAG_INSTALACAO', 'FPD', 'SCORE_01', 'SCORE_02', 'FLAG_SOS'
]

# Nome da coluna de safra
COLUNA_SAFRA = 'SAFRA'

for nome_df, df in dataframes.items():
    # 1. DAT_ para date
    colunas_data = [c for c in df.columns if c.startswith('DAT_')]
    for col_data in colunas_data:
        try:
            df = df.withColumn(col_data, to_date(col(col_data), 'dd/MM/yyyy'))
            total_dat += 1
        except:
            try:
                df = df.withColumn(col_data, to_date(col(col_data), 'yyyy-MM-dd'))
                total_dat += 1
            except:
                pass
    
    # 2. DATADENASCIMENTO para date
    if 'DATADENASCIMENTO' in df.columns:
        try:
            df = df.withColumn('DATADENASCIMENTO', to_date(col('DATADENASCIMENTO'), 'dd/MM/yyyy'))
            total_datadenascimento += 1
        except:
            try:
                df = df.withColumn('DATADENASCIMENTO', to_date(col('DATADENASCIMENTO'), 'yyyy-MM-dd'))
                total_datadenascimento += 1
            except:
                pass
    
    # 3. VAL_ para float
    colunas_valor = [c for c in df.columns if c.startswith('VAL_')]
    for col_valor in colunas_valor:
        try:
            df = df.withColumn(col_valor, col(col_valor).cast('float'))
            total_val += 1
        except:
            pass
    
    # 4. SAFRA (YYYYMM paraYYYY/MM)
    if COLUNA_SAFRA in df.columns:
        try:
            df = df.withColumn(
                COLUNA_SAFRA,
                concat(
                    substring(col(COLUNA_SAFRA), 1, 4),
                    lit('/'),
                    substring(col(COLUNA_SAFRA), 5, 2)
                )
            )
            total_safra += 1
        except:
            pass
    
    # 5. Colunas para integer
    for col_int in colunas_integer:
        if col_int in df.columns:
            try:
                df = df.withColumn(col_int, col(col_int).cast('integer'))
                total_int += 1
            except:
                pass
    
    dataframes[nome_df] = df

# Print final
print(f"DAT_ para date: {total_dat}")
print(f"DATADENASCIMENTO para date: {total_datadenascimento}")
print(f"VAL_ para float: {total_val}")
print(f"SAFRA para YYYY/MM: {total_safra}")
print(f"Colunas explícitas para integer: {total_int}")
print(f"Colunas nº inteiro: {colunas_integer}")
print(f"TOTAL CONVERTIDAS: {total_dat + total_datadenascimento + total_val + total_safra + total_int}")

### Análise univarida de variáveis numéricas

Estatísticas descritivas para colunas numéricas

In [ ]:
from pyspark.sql.functions import countDistinct
import pandas as pd


for nome_df, df in dataframes.items():
    # COLUNAS NUMÉRICAS 
    colunas_numericas = [f.name for f in df.schema.fields 
                         if 'Integer' in str(f.dataType) or 'Double' in str(f.dataType) or 'Long' in str(f.dataType or 'Float' in str(f.dataType))]
    
    if colunas_numericas:
        print(f"{nome_df.upper()}")
        
        # Descrever (count, mean, stddev, min, 25%, 50%, 75%, max)
        df_desc = df.select(colunas_numericas).describe()
        
        # Converter para pandas
        df_desc_pd = df_desc.toPandas().set_index('summary').T
        
        # Adicionar colunas extras (unique, top, freq)
        for col_name in colunas_numericas:
            unique = df.select(countDistinct(col_name)).collect()[0][0]
            top_freq = df.groupBy(col_name).count().orderBy('count', ascending=False).first()
            
            df_desc_pd.loc[col_name, 'unique'] = unique
            df_desc_pd.loc[col_name, 'top'] = top_freq[0]
            df_desc_pd.loc[col_name, 'freq'] = top_freq[1]
        
        # Reordenar colunas
        ordem = ['count', 'unique', 'top', 'freq', 'mean', 'stddev', 'min', '25%', '50%', '75%', 'max']
        ordem_final = [c for c in ordem if c in df_desc_pd.columns]
        
        display(df_desc_pd[ordem_final])
    else:
        print(f"\n{nome_df}: Nenhuma coluna numérica encontrada")

In [ ]:
Boxplot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.types import IntegerType, LongType, DoubleType, FloatType

def boxplots_var_num_spark(df_spark, nome_df=""):
    """
    Plota boxplots para colunas numéricas de um Spark DataFrame.
    
    :param df_spark: Spark DataFrame
    :param nome_df: Nome do dataframe
    """
    
    # IDENTIFICAR COLUNAS NUMÉRICAS
    colunas_numericas = [f.name for f in df_spark.schema.fields 
                         if isinstance(f.dataType, (IntegerType, LongType, DoubleType, FloatType))]
    
    if not colunas_numericas:
        print(f"Nenhuma coluna numérica em {nome_df}")
        return
    
    # CONVERTER PARA PANDAS
    df_pandas = df_spark.select(colunas_numericas).toPandas()
    
    # CRIAR PAINEL
    nrows = len(colunas_numericas) // 3 + (len(colunas_numericas) % 3 > 0)
    fig, axes = plt.subplots(nrows=nrows, ncols=3, figsize=(14, nrows * 4))
    
    if nrows == 1:
        axes = axes.reshape(1, -1)
    
    plt.tight_layout(pad=4)
    sns.set_style("whitegrid")
    
    # PLOTAR
    for i, column in enumerate(colunas_numericas):
        row = i // 3
        col = i % 3
        
        sns.boxplot(data=df_pandas[column], ax=axes[row, col], color="skyblue")
        axes[row, col].set_title(f'{column}', fontdict={'fontsize': 14, 'fontweight': 'bold'})
        axes[row, col].set_ylabel('')
    
    # REMOVER VAZIOS
    for j in range(len(colunas_numericas), nrows * 3):
        row = j // 3
        col = j % 3
        fig.delaxes(axes[row, col])
    
    fig.suptitle(f"Análise descritiva - BoxPlot - {nome_df}", fontsize=20, fontweight='bold', y=0.995)
    plt.show()

# EXECUTAR PARA TODOS DATAFRAMES
for nome_df, df in dataframes.items():
    boxplots_var_num_spark(df, nome_df)

Histograma

In [ ]:
def boxplots_var_num_spark(df_spark, nome_df=""):
    """
    Plota boxplots para colunas numéricas de um Spark DataFrame.
    
    :param df_spark: Spark DataFrame
    :param nome_df: Nome do dataframe
    """
    
    # IDENTIFICAR COLUNAS NUMÉRICAS
    colunas_numericas = [f.name for f in df_spark.schema.fields 
                         if isinstance(f.dataType, (IntegerType, LongType, DoubleType, FloatType))]
    
    if not colunas_numericas:
        print(f"Nenhuma coluna numérica em {nome_df}")
        return
    
    # CONVERTER PARA PANDAS
    df_pandas = df_spark.select(colunas_numericas).toPandas()
    
    # CRIAR PAINEL
    nrows = len(colunas_numericas) // 3 + (len(colunas_numericas) % 3 > 0)
    fig, axes = plt.subplots(nrows=nrows, ncols=3, figsize=(14, nrows * 4))
    
    if nrows == 1:
        axes = axes.reshape(1, -1)
    
    plt.tight_layout(pad=4)
    sns.set_style("whitegrid")
    
    # PLOTAR
    for i, column in enumerate(colunas_numericas):
        row = i // 3
        col = i % 3
        
        sns.boxplot(data=df_pandas[column], ax=axes[row, col], color="skyblue")
        axes[row, col].set_title(f'{column}', fontdict={'fontsize': 14, 'fontweight': 'bold'})
        axes[row, col].set_ylabel('')
    
    # REMOVER VAZIOS
    for j in range(len(colunas_numericas), nrows * 3):
        row = j // 3
        col = j % 3
        fig.delaxes(axes[row, col])
    
    fig.suptitle(f"Análise descritiva - BoxPlot - {nome_df}", fontsize=20, fontweight='bold', y=0.995)
    plt.show()

# EXECUTAR PARA TODOS DATAFRAMES
for nome_df, df in dataframes.items():
    boxplots_var_num_spark(df, nome_df)

### Análise univarida de variáveis categóricas

Gráfico de barras

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import gc
from pyspark.sql.types import StringType, DateType, BooleanType

def plotar_categoricas_por_lote(lista_nomes_df, dataframes_dict, metadata_df, 
                                corte_cardinalidade=30, graficos_por_linha=3, top_valores=None):
    """
    Plota gráficos de barras para colunas categóricas de um lote de dataframes.
    
    Parâmetros:
    - lista_nomes_df: Lista com nomes dos dataframes a plotar
    - dataframes_dict: Dicionário com todos os dataframes
    - metadata_df: DataFrame pandas com metadados (cardinalidade)
    - corte_cardinalidade: Cardinalidade máxima
    - graficos_por_linha: Quantidade de gráficos por linha
    - top_valores: Limitar a top N valores (None = todos)
    """
    
    for nome_df in lista_nomes_df:
        if nome_df not in dataframes_dict:
            print(f"{nome_df}: Dataframe não encontrado")
            continue
        
        df_spark = dataframes_dict[nome_df]
        
        # FILTRAR COLUNAS CATEGÓRICAS
        df_meta_filtrado = metadata_df[
            (metadata_df['nome_dataframe'] == nome_df) & 
            (metadata_df['cardinalidade'] <= corte_cardinalidade)
        ]
        colunas_selecionadas = df_meta_filtrado['nome_variavel'].tolist()
        
        if not colunas_selecionadas:
            print(f"{nome_df}: Nenhuma coluna categórica")
            continue
        
        # CONVERTER PARA PANDAS
        df_pandas = df_spark.select(colunas_selecionadas).toPandas()
        
        # FILTRAR COLUNAS COM DADOS
        colunas_validas = []
        for col_name in colunas_selecionadas:
            if df_pandas[col_name].notna().sum() > 0:
                colunas_validas.append(col_name)
        
        if not colunas_validas:
            print(f"{nome_df}: Nenhuma coluna com dados")
            del df_pandas
            gc.collect()
            continue
        
        print(f"{nome_df}: {len(colunas_validas)} colunas categóricas\n")
        
        # RIAR SUBPLOTS
        n_linhas = -(-len(colunas_validas) // graficos_por_linha)
        n_colunas = min(len(colunas_validas), graficos_por_linha)
        
        fig, axs = plt.subplots(nrows=n_linhas, ncols=n_colunas, figsize=(15, 4 * n_linhas))
        
        axs = np.atleast_2d(axs)
        if axs.shape[0] == 1 and axs.shape[1] > 1:
            axs = axs.reshape(n_linhas, n_colunas)
        
        sns.set_style("whitegrid")
        
        # PLOTAR
        for i, var in enumerate(colunas_validas):
            row_idx = i // graficos_por_linha
            col_idx = i % graficos_por_linha
            
            try:
                valor_counts = df_pandas[var].dropna().value_counts().sort_values(ascending=False)
                
                if len(valor_counts) == 0:
                    axs[row_idx, col_idx].text(0.5, 0.5, 'Sem dados', ha='center', va='center')
                    axs[row_idx, col_idx].set_title(f'{var}', fontdict={'fontsize': 12})
                    continue
                
                if top_valores:
                    valor_counts = valor_counts.head(top_valores)
                
                valor_counts.plot(kind='bar', ax=axs[row_idx, col_idx], color='skyblue')
                axs[row_idx, col_idx].set_title(f'{var}', fontdict={'fontsize': 12, 'fontweight': 'bold'})
                axs[row_idx, col_idx].set_ylabel('Frequência')
                axs[row_idx, col_idx].set_xlabel('')
                axs[row_idx, col_idx].tick_params(axis='x', rotation=45, labelsize=8)
            except Exception as e:
                print(f"{var}: {str(e)[:40]}")
        
        # REMOVER VAZIOS
        for j in range(len(colunas_validas), n_linhas * n_colunas):
            row_idx = j // graficos_por_linha
            col_idx = j % graficos_por_linha
            axs[row_idx, col_idx].axis('off')
        
        fig.suptitle(f"Análise Descritiva - Frequência Categórica - {nome_df}", 
                    fontsize=20, fontweight='bold', y=01.05)
        
        plt.tight_layout()
        display(fig)
        plt.close(fig)
        
        # LIBERAR MEMÓRIA
        del fig, axs, df_pandas
        gc.collect()

In [ ]:
# Converter metadados para pandas
if 'df_resultado' in locals():
    metadata_pandas = df_resultado.toPandas()
else:
    metadata_pandas = pd.DataFrame(metadata)

# Listar nomes de todos os dataframes
nomes_dfs = list(dataframes.keys())
print(f"Total de dataframes: {len(nomes_dfs)}")
print(f"Não serão plotados os dataframesnº 17 e 18\n")

for i, nome in enumerate(nomes_dfs):
    print(f"{i+1}. {nome}")

In [ ]:
# Primeiros 6 dataframes
print("LOTE 1: Primeiros 6 dataframes")

plotar_categoricas_por_lote(
    lista_nomes_df=nomes_dfs[0:6],  # dataframes 0-5
    dataframes_dict=dataframes,
    metadata_df=metadata_pandas,
    corte_cardinalidade=30,
    graficos_por_linha=3,
    top_valores=None
)

gc.collect()

In [ ]:
# Próximos 7 dataframes
print("LOTE 2: Dataframes 7-12")

plotar_categoricas_por_lote(
    lista_nomes_df=nomes_dfs[6:13],  # dataframes 6-12
    dataframes_dict=dataframes,
    metadata_df=metadata_pandas,
    corte_cardinalidade=30,
    graficos_por_linha=3,
    top_valores=None
)

gc.collect()

In [ ]:
# Próximos 3 dataframes
print("LOTE 3: Dataframes 13-16")

plotar_categoricas_por_lote(
    lista_nomes_df=nomes_dfs[13:16],  # Dataframes 13-16
    dataframes_dict=dataframes,
    metadata_df=metadata_pandas,
    corte_cardinalidade=30,
    graficos_por_linha=3,
    top_valores=None
)

gc.collect()

In [ ]:
# Último dataframe
print("LOTE 4: Dataframe 19")

plotar_categoricas_por_lote(
    lista_nomes_df=nomes_dfs[18:19],  # dataframe 19 (índice 18)
    dataframes_dict=dataframes,
    metadata_df=metadata_pandas,
    corte_cardinalidade=30,
    graficos_por_linha=3,
    top_valores=None
)

gc.collect()

### Análise multivariada de variáveis categóricas

#### Análise de variáveis numéricas

In [ ]:
def kdeplots_var_num_target_spark(df_spark, target_column='FPD', nome_df=""):
    """
    Plota gráficos kdeplot para variáveis numéricas discriminadas pelo target.
    
    :param df_spark: Spark DataFrame contendo as variáveis numéricas e a coluna target.
    :param target_column: Nome da coluna target (padrão: 'FPD').
    :param nome_df: Nome do DataFrame para exibição no título.
    """
    # Identifica colunas numéricas (excluindo target)
    numeric_columns = [f.name for f in df_spark.schema.fields 
                       if isinstance(f.dataType, (IntegerType, LongType, DoubleType, FloatType))
                       and f.name.lower() != target_column.lower()]
    
    if not numeric_columns:
        print(f"Nenhuma coluna numérica em {nome_df}")
        return
    
    # Converte para pandas
    dataframe = df_spark.toPandas()
    
    # Verifica se coluna target existe
    if target_column not in dataframe.columns:
        print(f"Coluna '{target_column}' não encontrada em {nome_df}")
        return
    
    # Define o número de linhas com base no número de colunas numéricas
    nrows = len(numeric_columns) // 3 + (len(numeric_columns) % 3 > 0)
    
    # Inicializa o painel de gráficos
    fig, axes = plt.subplots(nrows=nrows, ncols=3, figsize=(14, nrows * 4))
    
    # Tratamento para nrows == 1
    if nrows == 1:
        axes = axes.reshape(1, -1)
    
    # Ajusta o layout
    plt.tight_layout(pad=4)
    
    # Configura estilo e paleta de cores
    sns.set_style("whitegrid")
    
    # Plota kdeplots para cada coluna numérica, discriminando as curvas pelo valor da coluna target
    for i, column in enumerate(numeric_columns):
        row = i // 3
        col = i % 3
        
        sns.kdeplot(data=dataframe[dataframe[target_column] == 1][column], ax=axes[row, col], color="blue", label="1", fill=True, warn_singular=False)
        sns.kdeplot(data=dataframe[dataframe[target_column] == 0][column], ax=axes[row, col], color="red", label="0", fill=True, warn_singular=False)
        axes[row, col].set_title(f'{column}', fontdict={'fontsize': 14, 'fontweight': 'bold'})
        axes[row, col].set_ylabel('Densidade')
        axes[row, col].tick_params(axis='both', which='major', labelsize=12)
        if i == 0:
            axes[row, col].legend(title=target_column)
    
    # Remove gráficos vazios (se houver)
    for j in range(len(numeric_columns), nrows * 3):
        row = j // 3
        col = j % 3
        fig.delaxes(axes[row, col])
    
    # Adiciona título principal
    fig.suptitle(f"Análise descritiva - Gráfico KDE por Target - {nome_df}", fontsize=20, fontweight='bold', y=1.00)
    plt.show()


# EXECUTAR PARA TODOS OS DATAFRAMES COM FPD
for nome_df, df in dataframes.items():
    # Verifica se tem coluna FPD (case-insensitive)
    cols_lower = [f.name.lower() for f in df.schema.fields]
    if 'fpd' in cols_lower:
        # Encontra o nome exato da coluna
        col_fpd = [f.name for f in df.schema.fields if f.name.lower() == 'fpd'][0]
        kdeplots_var_num_target_spark(df, target_column=col_fpd, nome_df=nome_df)

#### Análise de variáveis categóricas

Gráficos de barras segmentado

In [ ]:
def plot_cat_vs_target_cutoff_spark(df_spark, target_column='FPD', cutoff=10, nome_df=""):
    """
    Plota gráficos de barras para analisar variáveis categóricas em relação ao target,
    limitando o número de variáveis de acordo com um valor de cutoff.

    :param df_spark: Spark DataFrame contendo as variáveis categóricas e a coluna target.
    :param target_column: Nome da coluna target (padrão: 'FPD').
    :param cutoff: Valor de cutoff para limitar o número de variáveis categóricas plotadas (padrão é 10).
    :param nome_df: Nome do DataFrame para exibição no título.
    """
    from pyspark.sql.types import StringType, BinaryType
    
    # Verifica se coluna target existe
    cols_spark = [f.name for f in df_spark.schema.fields]
    if target_column not in cols_spark:
        print(f"Coluna '{target_column}' não encontrada em {nome_df}")
        return
    
    # Identifica colunas categóricas (StringType)
    categorical_columns = [f.name for f in df_spark.schema.fields 
                          if isinstance(f.dataType, (StringType, BinaryType))]
    
    if not categorical_columns:
        print(f"Nenhuma coluna categórica em {nome_df}")
        return
    
    # Converte para pandas
    dataframe = df_spark.select(categorical_columns + [target_column]).toPandas()
    
    # Filtra as colunas com base no cutoff
    categorical_columns_filtered = [col for col in categorical_columns 
                                   if dataframe[col].nunique() <= cutoff]
    
    if not categorical_columns_filtered:
        print(f"Nenhuma coluna categórica passa no cutoff ({cutoff}) em {nome_df}")
        return
    
    # Define o número de linhas e colunas para os subplots
    n_cols = 3
    n_rows = len(categorical_columns_filtered) // n_cols + (len(categorical_columns_filtered) % n_cols > 0)
    
    # Cria subplots
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 5))
    
    # Tratamento para nrows == 1
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    
    # Ajusta o layout
    plt.tight_layout(pad=4)
    
    # Loop pelas colunas categóricas filtradas
    for i, column in enumerate(categorical_columns_filtered):
        row = i // n_cols
        col = i % n_cols
        ax = axes[row, col]
        
        # Calcula proporções de cada categoria para cada valor do target
        prop_df = (dataframe.groupby([column, target_column]).size() / dataframe.groupby(target_column).size()).unstack()
        
        # Plota o gráfico de barras
        prop_df.plot(kind='bar', stacked=True, ax=ax)
        ax.set_title(column, fontsize=14, fontweight='bold')
        ax.set_ylabel('Proporção')
        ax.set_xlabel('')
        ax.tick_params(axis='both', which='major', labelsize=12)
        
        # Rotaciona as labels do eixo x
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
        
        # Ajusta legenda
        ax.legend(title=target_column, fontsize=10)
    
    # Remove subplots vazios
    for j in range(len(categorical_columns_filtered), n_rows * n_cols):
        row = j // n_cols
        col = j % n_cols
        fig.delaxes(axes[row, col])
    
    # Adiciona título principal
    fig.suptitle(f"Análise de Variáveis Categóricas em relação ao Target - {nome_df}", 
                 fontsize=16, fontweight='bold', y=0.995)
    plt.show()


# EXECUTAR PARA TODOS OS DATAFRAMES COM FPD
for nome_df, df in dataframes.items():
    # Verifica se tem coluna FPD (case-insensitive)
    cols_lower = [f.name.lower() for f in df.schema.fields]
    if 'fpd' in cols_lower:
        # Encontra o nome exato da coluna
        col_fpd = [f.name for f in df.schema.fields if f.name.lower() == 'fpd'][0]
        print(f"\nProcessando {nome_df}...\n")
        plot_cat_vs_target_cutoff_spark(df, target_column=col_fpd, cutoff=10, nome_df=nome_df)

#### Análise do público

In [ ]:
def pod_count_categorias_spark(df_spark, columns):
    """
    Calcula contagem de valores e porcentagem para variáveis categóricas em Spark DataFrame.
    
    :param df_spark: Spark DataFrame
    :param columns: Nome da coluna ou lista de colunas
    :return: Pandas DataFrame com contagem, porcentagem e total
    """
    from pyspark.sql.functions import col
    
    if isinstance(columns, str):
        columns = [columns]
    
    # Seleciona as colunas e converte para string para evitar problemas de tipo
    df_selected = df_spark.select([col(c).cast('string').alias(c) for c in columns])
    
    # Calcula a contagem de valores usando groupBy
    count_df = df_selected.groupBy(columns).count()
    
    # Converte para pandas
    count_df_pandas = count_df.toPandas()
    
    # Renomeia coluna 'count' para 'Count'
    count_df_pandas = count_df_pandas.rename(columns={'count': 'Count'})
    
    # Calcula a porcentagem de cada valor
    count_df_pandas['Percentage'] = (count_df_pandas['Count'] / count_df_pandas['Count'].sum()) * 100
    
    # Calcula a soma total
    total_count = count_df_pandas['Count'].sum()
    
    # Cria dicionário para a linha total
    total_row_dict = {col: 'Total' for col in columns}
    total_row_dict['Count'] = total_count
    total_row_dict['Percentage'] = 100.0
    
    # Adiciona a linha total
    total_row = pd.DataFrame([total_row_dict])
    count_df_pandas = pd.concat([count_df_pandas, total_row], ignore_index=True)
    
    return count_df_pandas

Quantidade de inadimplentes vs adimplentes

In [ ]:
# Registrar as 4 views
bases_com_fpd = ['base_dados_cadastrais', 'base_score_bureau_movel', 
                 'base_score_bureau_movel_full', 'base_telco']

for nome_base in bases_com_fpd:
    dataframes[nome_base].createOrReplaceTempView(f'{nome_base}_view')

# Executar UNION de todas as bases
resultado = spark.sql("""
    SELECT 'base_dados_cadastrais' as Base, FPD, Count, Percentage FROM (
        SELECT 
            CAST(FPD AS STRING) as FPD,
            COUNT(*) as Count,
            ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as Percentage
        FROM base_dados_cadastrais_view
        GROUP BY FPD
        
        UNION ALL
        
        SELECT 'TOTAL', COUNT(*), 100.0 FROM base_dados_cadastrais_view
    )
    
    UNION ALL
    
    SELECT 'base_score_bureau_movel' as Base, FPD, Count, Percentage FROM (
        SELECT 
            CAST(FPD AS STRING) as FPD,
            COUNT(*) as Count,
            ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as Percentage
        FROM base_score_bureau_movel_view
        GROUP BY FPD
        
        UNION ALL
        
        SELECT 'TOTAL', COUNT(*), 100.0 FROM base_score_bureau_movel_view
    )
    
    UNION ALL
    
    SELECT 'base_score_bureau_movel_full' as Base, FPD, Count, Percentage FROM (
        SELECT 
            CAST(FPD AS STRING) as FPD,
            COUNT(*) as Count,
            ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as Percentage
        FROM base_score_bureau_movel_full_view
        GROUP BY FPD
        
        UNION ALL
        
        SELECT 'TOTAL', COUNT(*), 100.0 FROM base_score_bureau_movel_full_view
    )
    
    UNION ALL
    
    SELECT 'base_telco' as Base, FPD, Count, Percentage FROM (
        SELECT 
            CAST(FPD AS STRING) as FPD,
            COUNT(*) as Count,
            ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as Percentage
        FROM base_telco_view
        GROUP BY FPD
        
        UNION ALL
        
        SELECT 'TOTAL', COUNT(*), 100.0 FROM base_telco_view
    )
    
    ORDER BY Base, CASE WHEN FPD = 'TOTAL' THEN 1 ELSE 0 END, FPD
""")

display(resultado)

Análise por produto da carteira

In [ ]:
dataframes['base_dados_cadastrais'].createOrReplaceTempView('base_dados_cadastrais_view')

resultado = spark.sql("""
    SELECT 
        CAST(PROD AS STRING) as PROD,
        COUNT(*) as Count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as Percentage
    FROM base_dados_cadastrais_view
    GROUP BY PROD
    
    UNION ALL
    
    SELECT 
        'TOTAL',
        COUNT(*),
        100.0
    FROM base_dados_cadastrais_view
    
    ORDER BY 
        CASE WHEN PROD = 'TOTAL' THEN 1 ELSE 0 END,
        PROD
""")

display(resultado)

In [ ]:
resultado = pod_count_categorias_spark(dataframes['base_dados_cadastrais'], ['PROD', 'FPD'])
display(resultado)

Ocupação

In [ ]:
dataframes['base_dados_cadastrais'].createOrReplaceTempView('base_dados_cadastrais_view')

resultado = spark.sql("""
    SELECT 
        CAST(var_25 AS STRING) as var_25,
        COUNT(*) as Count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as Percentage
    FROM base_dados_cadastrais_view
    GROUP BY var_25
    
    UNION ALL
    
    SELECT 
        'TOTAL',
        COUNT(*),
        100.0
    FROM base_dados_cadastrais_view
    
    ORDER BY 
        CASE WHEN var_25 = 'TOTAL' THEN 1 ELSE 0 END,
        var_25
""")

display(resultado)

In [ ]:
resultado = pod_count_categorias_spark(dataframes['base_dados_cadastrais'], ['var_25', 'FPD'])
display(resultado)

Situação do CPF na Receita Federal

In [ ]:
dataframes['base_dados_cadastrais'].createOrReplaceTempView('base_dados_cadastrais_view')

resultado = spark.sql("""
    SELECT 
        CAST(STATUSRF AS STRING) as STATUSRF,
        COUNT(*) as Count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as Percentage
    FROM base_dados_cadastrais_view
    GROUP BY STATUSRF
    
    UNION ALL
    
    SELECT 
        'TOTAL',
        COUNT(*),
        100.0
    FROM base_dados_cadastrais_view
    
    ORDER BY 
        CASE WHEN STATUSRF = 'TOTAL' THEN 1 ELSE 0 END,
        STATUSRF
""")

display(resultado)

In [ ]:
resultado = pod_count_categorias_spark(dataframes['base_dados_cadastrais'], ['STATUSRF', 'FPD'])
display(resultado)